# <font color="#418FDE" size="6.5" uppercase>**Autoencoder und Generierung**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Unterscheiden diskriminative und generative Aufgaben anhand kleiner Datenbeispiele. 
- Trainieren kleine Autoencoder zur Rekonstruktion von Digits- oder MNIST-Teilmengen. 
- Analysieren Rekonstruktionsfehler, Denoising, latente Interpolation und Grenzen generativer Ansätze. 


## **1. Autoencoder Idee**

### **1.1. Diskriminativ oder generativ**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_01_01.jpg?v=1787667371" width="250">



>* Diskriminative Modelle entscheiden zwischen vorhandenen Kategorien
>* Merkmale helfen, klare Klassengrenzen zu lernen

>* Generative Modelle lernen Datenmuster und Strukturen.
>* Sie rekonstruieren, ergänzen oder erzeugen plausible Beispiele.

>* Autoencoder rekonstruieren Daten aus komprimierten Darstellungen
>* Nützlich, aber keine vollständigen Generierungsmodelle



In [ ]:
#@title Python-Code - Diskriminativ oder generativ

# Dieses Beispiel vergleicht zwei Lernziele.
# Diskriminativ bedeutet entscheiden, generativ bedeutet rekonstruieren.
# Die Grafik zeigt Klassen und Rekonstruktionen.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import sklearn

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()

# Wir nutzen nur zwei Klassen für klare Anschauung.
selected = digits.target < 2
X = digits.data[selected]
y = digits.target[selected]

# Eine einfache Prüfung macht die Datenannahme sichtbar.
if X.shape[0] == 0 or X.shape[1] != 64:
    raise ValueError("Die Zifferndaten haben nicht die erwartete Form.")

# Der Split trennt Lernen und Prüfen fair.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Dieses Modell lernt eine Grenze zwischen Null und Eins.
classifier = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=300, random_state=42)
)

# PCA rekonstruiert Bilder aus wenigen latenten Zahlen.
reconstructor = make_pipeline(StandardScaler(), PCA(n_components=8, random_state=42))

# Beide Modelle sehen dieselben Trainingsbilder.
classifier.fit(X_train, y_train)
reconstructor.fit(X_train)

# Jetzt vergleichen wir Entscheidung und Rekonstruktion.
predicted = classifier.predict(X_test)
encoded = reconstructor.transform(X_test)
reconstructed = reconstructor.inverse_transform(encoded)

# Der Rekonstruktionsfehler misst Bildähnlichkeit statt Klassenrichtigkeit.
accuracy = accuracy_score(y_test, predicted)
reconstruction_error = np.mean((X_test - reconstructed) ** 2)

# Wir wählen ein Testbild für die sichtbare Rekonstruktion.
example_index = 0
original_image = X_test[example_index].reshape(8, 8)
reconstructed_image = reconstructed[example_index].reshape(8, 8)

# Die Ausgabe benennt die zwei unterschiedlichen Aufgaben.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Diskriminativ: Genauigkeit Null gegen Eins = {accuracy:.2f}")
print(f"Generativ gedacht: mittlerer Rekonstruktionsfehler = {reconstruction_error:.2f}")
print(f"Beispiel: wahr {y_test[example_index]}, vorhergesagt {predicted[example_index]}")

# Ein Bild zeigt Original und Rekonstruktion nebeneinander.
combined_image = np.concatenate([original_image, reconstructed_image], axis=1)

# Die Achse zeigt Pixelpositionen im kombinierten Bild.
fig, ax = plt.subplots(figsize=(5, 2.5))
ax.imshow(combined_image, cmap="gray_r", interpolation="nearest")
ax.set_title("Links Original, rechts Rekonstruktion")
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")
ax.axvline(7.5, color="red", linewidth=2)
plt.show()



### **1.2. Encoder und Decoder**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_01_02.jpg?v=1787667374" width="250">



>* Encoder verdichtet Eingaben zu wichtigen Merkmalen
>* Decoder rekonstruiert daraus ähnliche Daten

>* Diskriminative Modelle entscheiden Kategorien.
>* Autoencoder verdichten und rekonstruieren wichtige Strukturen.

>* Decoder erzeugen Daten aus kompakten Beschreibungen
>* Einfache Autoencoder rekonstruieren oft nur begrenzt



In [ ]:
#@title Python-Code - Encoder und Decoder

# Wir zerlegen Ziffernbilder in Encoder und Decoder.
# Der Engpass zeigt eine kompakte innere Beschreibung.
# Rekonstruktion unterscheidet sich sichtbar von Klassifikation.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
import sklearn

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images.astype(float) / 16.0
labels = digits.target

# Eine einfache Prüfung macht die Datenannahme sichtbar.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden kleine 8-mal-8-Ziffernbilder.")

# Der Encoder verdichtet jedes Bild auf zwei Zahlen.
flat_images = images.reshape(len(images), 64)
encoder_decoder = PCA(n_components=2, random_state=42)
latent_codes = encoder_decoder.fit_transform(flat_images)

# Der Decoder rekonstruiert Bilder aus diesen zwei Zahlen.
reconstructed_flat = encoder_decoder.inverse_transform(latent_codes)
reconstruction_error = mean_squared_error(flat_images, reconstructed_flat)

# Wir vergleichen Rekonstruktion mit einer diskriminativen Ausgabe.
example_index = 0
true_label = labels[example_index]
latent_pair = latent_codes[example_index]

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Diskriminative Ausgabe wäre nur die Klasse: {true_label}")
print(f"Encoder-Ausgabe: zwei latente Zahlen {np.round(latent_pair, 2)}")
print(f"Mittlerer Rekonstruktionsfehler: {reconstruction_error:.3f}")
print("Decoder-Ausgabe: ein rekonstruiertes Bild statt einer Klasse.")

# Ein Mischbild zeigt die generative Rolle des Decoders.
zero_index = int(np.where(labels == 0)[0][0])
one_index = int(np.where(labels == 1)[0][0])

# Wir interpolieren zwischen zwei inneren Beschreibungen.
mixed_code = 0.5 * latent_codes[zero_index] + 0.5 * latent_codes[one_index]
mixed_image = encoder_decoder.inverse_transform([mixed_code])[0].reshape(8, 8)

# Die einzelne Grafik zeigt ein vom Decoder erzeugtes Bild.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(mixed_image, cmap="gray", vmin=0.0, vmax=1.0)
ax.set_title("Decoder aus gemischtem latentem Code")
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")
plt.show()



### **1.3. Latenter Raum**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_01_03.jpg?v=1787667376" width="250">



>* Latenter Raum verdichtet wichtige Bildmerkmale.
>* Nahe Punkte bedeuten ähnliche Eingaben.

>* Diskriminative Modelle trennen Klassen durch Grenzen
>* Autoencoder ordnen Daten nach ähnlichen Merkmalen

>* Interpolation ermöglicht schrittweise neue Rekonstruktionen
>* Unbekannte Bereiche liefern oft schlechte Ergebnisse



## **2. Rekonstruktion trainieren**

### **2.1. Dichter Autoencoder**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_02_01.jpg?v=1787667378" width="250">



>* Encoder verdichtet Bilder, Decoder rekonstruiert sie
>* Lernt typische Ziffernmuster ohne Klassenlabels

>* Dichte Schichten verbinden alle Einheiten vollständig
>* Latenter Raum erzwingt wichtige Bildmerkmale

>* Rekonstruktionen werden beim Training zunehmend ziffernähnlich
>* Gute Kompression vermeidet Auswendiglernen und Übervereinfachung



In [ ]:
#@title Python-Code - Dichter Autoencoder

# Wir trainieren einen kleinen dichten Autoencoder.
# Die Engstelle lernt eine kompakte Zifferndarstellung.
# Rekonstruktionen werden als Fehlerkurve sichtbar.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images

# Die Pixelwerte werden auf den Bereich null bis eins skaliert.
flat_images = images.reshape(len(images), -1) / 16.0

# Training und Test bleiben getrennt.
train_images, test_images = train_test_split(
    flat_images, test_size=0.25, random_state=42
)

# Diese Prüfung macht die erwartete Bildform ausdrücklich sichtbar.
if flat_images.shape[1] != 64:
    raise ValueError("Erwartet wurden 64 Pixel pro Ziffernbild.")

# Ein MLP mit kleiner Mittelschicht wirkt als dichter Autoencoder.
autoencoder = MLPRegressor(
    hidden_layer_sizes=(32, 8, 32), activation="relu", solver="adam",
    max_iter=120, random_state=42, verbose=False
)

# Zielwerte sind identisch mit den Eingaben.
autoencoder.fit(train_images, train_images)

# Das trainierte Netz rekonstruiert unbekannte Testbilder.
reconstructed = autoencoder.predict(test_images)
reconstructed = np.clip(reconstructed, 0.0, 1.0)

# Der mittlere quadratische Fehler misst die Rekonstruktionsqualität.
train_error = mean_squared_error(train_images, autoencoder.predict(train_images))
test_error = mean_squared_error(test_images, reconstructed)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsfehler: {train_error:.4f}")
print(f"Testfehler: {test_error:.4f}")
print("Kleiner Fehler bedeutet: Ausgabe ähnelt Eingabe stärker.")

# Wir zeigen ein Original und seine Rekonstruktion nebeneinander.
original_image = test_images[0].reshape(8, 8)
reconstructed_image = reconstructed[0].reshape(8, 8)
combined_image = np.concatenate([original_image, reconstructed_image], axis=1)

fig, ax = plt.subplots(figsize=(5, 2.5))
ax.imshow(combined_image, cmap="gray", vmin=0, vmax=1)
ax.set_title("Links Original, rechts Rekonstruktion")
ax.set_xlabel("Pixelspalten")
ax.set_ylabel("Pixelzeilen")
ax.set_xticks([])
ax.set_yticks([])
plt.show()



### **2.2. Rekonstruktionsverlust messen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_02_02.jpg?v=1787667382" width="250">



>* Verlust vergleicht Original und Rekonstruktion.
>* Niedriger Verlust zeigt gelernte Ziffernmuster.

>* Pixelabweichungen passend zur Bilddarstellung messen
>* Verlustwerte immer visuell einordnen

>* Trainings- und Validierungsverlust gemeinsam beobachten
>* Verlustkurven zeigen Lernen oder Überanpassung



In [ ]:
#@title Python-Code - Rekonstruktionsverlust messen

# Wir messen Rekonstruktionsverlust bei kleinen Ziffernbildern.
# Ein Autoencoder lernt komprimierte Bilddarstellungen.
# Die Kurve zeigt Training und Validierung.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images

# Diese Prüfung macht die Bildannahme sichtbar.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden kleine Bilder mit 8 mal 8 Pixeln.")

# Pixelwerte werden auf den Bereich null bis eins skaliert.
features = images.reshape(len(images), 64).astype("float32") / 16.0

# Der Autoencoder soll Eingaben als Ziel rekonstruieren.
X_train, X_valid = train_test_split(
    features, test_size=0.25, random_state=42
)

# Ein kleines Netz bildet Encoder und Decoder gemeinsam.
autoencoder = MLPRegressor(
    hidden_layer_sizes=(16,), activation="relu", solver="adam",
    max_iter=1, warm_start=True, random_state=42
)

train_losses = []
valid_losses = []

# Jede Epoche verbessert die Rekonstruktion schrittweise.
for epoch in range(30):
    autoencoder.fit(X_train, X_train)
    train_prediction = autoencoder.predict(X_train)
    valid_prediction = autoencoder.predict(X_valid)
    train_losses.append(mean_squared_error(X_train, train_prediction))
    valid_losses.append(mean_squared_error(X_valid, valid_prediction))

# Wir geben nur wenige zentrale Kennzahlen aus.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Trainingsverlust am Anfang: {train_losses[0]:.4f}")
print(f"Trainingsverlust am Ende: {train_losses[-1]:.4f}")
print(f"Validierungsverlust am Ende: {valid_losses[-1]:.4f}")

# Die Kurven zeigen Lernen und mögliche Überanpassung.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, 31), train_losses, label="Training")
ax.plot(range(1, 31), valid_losses, label="Validierung")
ax.set_title("Rekonstruktionsverlust eines kleinen Autoencoders")
ax.set_xlabel("Epoche")
ax.set_ylabel("Mittlerer quadratischer Fehler")
ax.legend()
plt.show()



### **2.3. Originale vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_02_03.jpg?v=1787667380" width="250">



>* Rekonstruktionen zeigen gelernte Bildmerkmale
>* Visueller Vergleich ergänzt den Rekonstruktionsverlust

>* Unschärfe und Formfehler getrennt betrachten
>* Wichtige Merkmale müssen erhalten bleiben

>* Vergleiche zeigen Grenzen und typische Muster
>* Rekonstruktionen verraten die gelernte Kompression



In [ ]:
#@title Python-Code - Originale vergleichen

# Wir vergleichen Originale mit Autoencoder-Rekonstruktionen.
# Der Rekonstruktionsfehler zeigt verlorene Bilddetails.
# Die Grafik markiert typische und schwierige Ziffern.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images.astype(np.float32) / 16.0

# Jedes Bild wird zu einem Vektor umgeformt.
flat_images = images.reshape(images.shape[0], -1)
labels = digits.target

# Diese Prüfung schützt vor unerwarteten Datenformen.
if flat_images.shape[1] != 64:
    raise ValueError("Erwartet werden 8-mal-8-Ziffernbilder.")

# Trainings- und Testdaten bleiben sauber getrennt.
train_x, test_x, train_y, test_y = train_test_split(
    flat_images, labels, test_size=0.25, random_state=42, stratify=labels
)

# Ein kleiner Autoencoder lernt, Eingaben selbst zu rekonstruieren.
autoencoder = MLPRegressor(
    hidden_layer_sizes=(16,), activation="relu", solver="adam",
    max_iter=120, random_state=42, verbose=False
)

autoencoder.fit(train_x, train_x)
reconstructed = autoencoder.predict(test_x)
reconstructed = np.clip(reconstructed, 0.0, 1.0)

# Pro Testbild messen wir den mittleren quadratischen Fehler.
errors = np.mean((test_x - reconstructed) ** 2, axis=1)
best_index = int(np.argmin(errors))
worst_index = int(np.argmax(errors))

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Mittlerer Test-Rekonstruktionsfehler: {mean_squared_error(test_x, reconstructed):.4f}")
print(f"Bestes Beispiel: Ziffer {test_y[best_index]}, Fehler {errors[best_index]:.4f}")
print(f"Schwierigstes Beispiel: Ziffer {test_y[worst_index]}, Fehler {errors[worst_index]:.4f}")

# Vier kleine Bilder werden zu einer Vergleichsfläche zusammengesetzt.
comparison = np.zeros((18, 18), dtype=np.float32)
comparison[0:8, 0:8] = test_x[best_index].reshape(8, 8)
comparison[0:8, 10:18] = reconstructed[best_index].reshape(8, 8)
comparison[10:18, 0:8] = test_x[worst_index].reshape(8, 8)
comparison[10:18, 10:18] = reconstructed[worst_index].reshape(8, 8)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(comparison, cmap="gray", vmin=0.0, vmax=1.0)

ax.set_title("Originale links, Rekonstruktionen rechts")
ax.set_xlabel("Spalten im Vergleichsbild")
ax.set_ylabel("Oben: kleiner Fehler, unten: großer Fehler")

ax.set_xticks([])
ax.set_yticks([])
plt.show()



## **3. Denoising und Grenzen**

### **3.1. Rauschen entfernen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_03_01.jpg?v=1787667383" width="250">



>* Verrauschte Eingaben zu sauberen Signalen rekonstruieren
>* Latenter Engpass speichert Wesentliches, verwirft Störungen

>* Denoising erzeugt strukturierte, plausiblere Signale
>* Es funktioniert nur bei bekannten Störungen

>* Denoising kann seltene wichtige Details entfernen
>* Bewerte Fehler, Beispiele und Datenkontext gemeinsam



In [ ]:
#@title Python-Code - Rauschen entfernen

# Wir entfernen Rauschen aus kleinen Ziffernbildern.
# Ein Autoencoder lernt stabile Bildstrukturen.
# Der Fehler zeigt Nutzen und Grenzen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images.astype("float32") / 16.0

# Diese Prüfung macht die Bildannahme sichtbar.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden kleine 8-mal-8-Ziffernbilder.")

# Wir formen jedes Bild zu einer Pixelzeile um.
clean_data = images.reshape(images.shape[0], -1)
labels = digits.target

# Der Split trennt Training und spätere Bewertung.
train_clean, test_clean, train_labels, test_labels = train_test_split(
    clean_data, labels, test_size=0.25, random_state=42, stratify=labels
)

# Zufälliges Pixelrauschen simuliert gestörte Eingaben.
rng = np.random.default_rng(42)
train_noise = rng.normal(0.0, 0.35, train_clean.shape)
test_noise = rng.normal(0.0, 0.35, test_clean.shape)

# Werte bleiben im gültigen Graustufenbereich.
train_noisy = np.clip(train_clean + train_noise, 0.0, 1.0)
test_noisy = np.clip(test_clean + test_noise, 0.0, 1.0)

# Der Autoencoder rekonstruiert saubere Bilder aus verrauschten Eingaben.
autoencoder = MLPRegressor(
    hidden_layer_sizes=(32, 12, 32), activation="relu", max_iter=120,
    random_state=42, verbose=False, early_stopping=True
)

# Eingabe ist verrauscht, Ziel ist sauber.
autoencoder.fit(train_noisy, train_clean)
reconstructed = np.clip(autoencoder.predict(test_noisy), 0.0, 1.0)

# Wir vergleichen Fehler vor und nach dem Denoising.
noisy_mse = mean_squared_error(test_clean, test_noisy)
denoised_mse = mean_squared_error(test_clean, reconstructed)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Mittlerer Fehler verrauscht: {noisy_mse:.4f}")
print(f"Mittlerer Fehler rekonstruiert: {denoised_mse:.4f}")

# Ein einzelnes Bild zeigt die Grenze der Rekonstruktion.
example_index = 0
clean_image = test_clean[example_index].reshape(8, 8)
noisy_image = test_noisy[example_index].reshape(8, 8)
denoised_image = reconstructed[example_index].reshape(8, 8)

# Drei kleine Bilder werden nebeneinander in einer Achse kombiniert.
separator = np.ones((8, 1))
combined_image = np.hstack((clean_image, separator, noisy_image, separator, denoised_image))

fig, ax = plt.subplots(figsize=(7, 2.4))
ax.imshow(combined_image, cmap="gray", vmin=0.0, vmax=1.0)
ax.set_title("Sauber | verrauscht | rekonstruiert")
ax.set_xlabel("Pixelpositionen in drei Ansichten")
ax.set_ylabel("Pixelzeilen")
ax.set_xticks([])
ax.set_yticks([])
plt.show()



### **3.2. Anomalien erkennen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_03_02.jpg?v=1787667386" width="250">



>* Autoencoder lernen normale Muster zu rekonstruieren
>* Große Rekonstruktionsfehler deuten auf Anomalien

>* Nützlich bei vielen Normaldaten, wenigen Fehlerbeispielen
>* Rekonstruktionsfehler immer vorsichtig und kontextbezogen deuten

>* Autoencoder verstehen Abweichungen nicht semantisch
>* Fehlerwerte brauchen Kontext und Fachprüfung



In [ ]:
#@title Python-Code - Anomalien erkennen

# Wir erkennen Anomalien mit Rekonstruktionsfehlern.
# Ein Autoencoder lernt normale Ziffernmuster.
# Hohe Fehler markieren ungewöhnliche Eingaben.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images.astype(np.float32) / 16.0

# Diese Prüfung macht die Bildannahme sichtbar.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden kleine 8-mal-8-Ziffernbilder.")

# Normale Trainingsdaten enthalten nur Ziffern ohne künstliche Störung.
flat_images = images.reshape(len(images), -1)
train_data, test_data = train_test_split(
    flat_images, test_size=0.25, random_state=42
)

# Der Autoencoder rekonstruiert seine Eingabe als Zielwert.
autoencoder = MLPRegressor(
    hidden_layer_sizes=(16,), activation="relu", max_iter=300, random_state=42
)

autoencoder.fit(train_data, train_data)

# Wir erzeugen deterministische Anomalien durch helle Bildflecken.
rng = np.random.default_rng(42)
normal_sample = test_data[:120].copy()
anomaly_sample = normal_sample.copy()

# Jede Anomalie bekommt einen kleinen hellen Block.
for row in range(len(anomaly_sample)):
    start = rng.integers(0, 6)
    anomaly_image = anomaly_sample[row].reshape(8, 8)
    anomaly_image[start:start + 3, start:start + 3] = 1.0

# Rekonstruktionsfehler messen den Abstand zur gelernten Normalität.
normal_recon = autoencoder.predict(normal_sample)
anomaly_recon = autoencoder.predict(anomaly_sample)

normal_errors = np.mean((normal_sample - normal_recon) ** 2, axis=1)
anomaly_errors = np.mean((anomaly_sample - anomaly_recon) ** 2, axis=1)

# Ein einfacher Schwellenwert kommt nur aus normalen Testfehlern.
threshold = np.percentile(normal_errors, 95)
normal_flag_rate = np.mean(normal_errors > threshold)
anomaly_flag_rate = np.mean(anomaly_errors > threshold)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Mittlerer Fehler normal: {normal_errors.mean():.4f}")
print(f"Mittlerer Fehler anomal: {anomaly_errors.mean():.4f}")
print(f"Schwellenwert, 95. Perzentil normal: {threshold:.4f}")
print(f"Markierte normale Bilder: {normal_flag_rate:.1%}")
print(f"Markierte anomale Bilder: {anomaly_flag_rate:.1%}")

# Das Histogramm zeigt die Trennung der Fehlerverteilungen.
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(normal_errors, bins=18, alpha=0.7, label="normale Ziffern")
ax.hist(anomaly_errors, bins=18, alpha=0.7, label="künstliche Anomalien")
ax.axvline(threshold, color="black", linestyle="--", label="Schwellenwert")
ax.set_title("Anomalieerkennung über Rekonstruktionsfehler")
ax.set_xlabel("Mittlerer quadratischer Rekonstruktionsfehler")
ax.set_ylabel("Anzahl Bilder")
ax.legend()
plt.show()



### **3.3. Eigenes Autoencoder Projekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_03_03.jpg?v=1787667388" width="250">



>* Klare Fragestellung für Rekonstruktion und Störungen
>* Rekonstruktionsfehler im Anwendungskontext bewerten

>* Rekonstruktion, Denoising und Anomalien klar trennen
>* Rekonstruktionsfehler systematisch und kritisch prüfen

>* Latenten Raum durch Interpolation untersuchen
>* Generative Wirkung kritisch und verantwortungsvoll bewerten



In [ ]:
#@title Python-Code - Eigenes Autoencoder Projekt

# Dieses Projekt prüft einen kleinen Denoising-Autoencoder.
# Rekonstruktionsfehler zeigen normale und gestörte Ziffern.
# Die Grafik macht Grenzen des Modells sichtbar.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images
labels = digits.target

# Diese Prüfung schützt vor unerwarteten Datenformen.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden kleine 8-mal-8-Ziffernbilder.")

# Pixelwerte werden für das neuronale Netz skaliert.
flat_images = images.reshape(len(images), 64)
scaler = MinMaxScaler()
clean_data = scaler.fit_transform(flat_images)

# Wir trennen Trainingsdaten und Testdaten sauber.
train_clean, test_clean, train_labels, test_labels = train_test_split(
    clean_data, labels, test_size=0.25, random_state=42, stratify=labels
)

# Rauschen simuliert eine typische Denoising-Aufgabe.
rng = np.random.default_rng(42)
train_noise = rng.normal(0.0, 0.25, train_clean.shape)
test_noise = rng.normal(0.0, 0.25, test_clean.shape)

# Die verrauschten Eingaben bleiben im gültigen Pixelbereich.
train_noisy = np.clip(train_clean + train_noise, 0.0, 1.0)
test_noisy = np.clip(test_clean + test_noise, 0.0, 1.0)

# Ein kleiner Autoencoder rekonstruiert saubere Zielbilder.
autoencoder = MLPRegressor(
    hidden_layer_sizes=(32, 8, 32),
    activation="relu",
    solver="adam",
    max_iter=120,
    random_state=42,
    verbose=False,
)

# Das Modell lernt von verrauschten zu sauberen Bildern.
autoencoder.fit(train_noisy, train_clean)
reconstructed = np.clip(autoencoder.predict(test_noisy), 0.0, 1.0)

# Rekonstruktionsfehler messen die mittlere quadratische Abweichung.
noisy_error = np.mean((test_noisy - test_clean) ** 2, axis=1)
recon_error = np.mean((reconstructed - test_clean) ** 2, axis=1)

# Ein künstlicher Fleck zeigt eine mögliche Modellgrenze.
spot_data = test_clean.copy()
spot_data[:, 18:22] = 1.0
spot_reconstructed = np.clip(autoencoder.predict(spot_data), 0.0, 1.0)
spot_error = np.mean((spot_reconstructed - test_clean) ** 2, axis=1)

# Wir wählen ein Beispiel mit klarer Verbesserung.
improvement = noisy_error - recon_error
example_index = int(np.argmax(improvement))

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Testbilder: {len(test_clean)}")
print(f"Fehler verrauscht: {np.mean(noisy_error):.4f}")
print(f"Fehler rekonstruiert: {np.mean(recon_error):.4f}")
print(f"Fehler mit künstlichem Fleck: {np.mean(spot_error):.4f}")
print(f"Beispielziffer in der Grafik: {test_labels[example_index]}")

# Eine Achse zeigt Original, Störung und Rekonstruktion nebeneinander.
comparison = np.hstack(
    [
        test_clean[example_index].reshape(8, 8),
        test_noisy[example_index].reshape(8, 8),
        reconstructed[example_index].reshape(8, 8),
        spot_data[example_index].reshape(8, 8),
        spot_reconstructed[example_index].reshape(8, 8),
    ]
)

fig, ax = plt.subplots(figsize=(9, 2.2))
ax.imshow(comparison, cmap="gray", vmin=0.0, vmax=1.0)
ax.set_title("Original, Rauschen, Denoising und Grenze")
ax.set_xlabel("Fünf Bildvarianten nebeneinander")
ax.set_ylabel("Pixelzeile")
ax.set_xticks([3.5, 11.5, 19.5, 27.5, 35.5])
ax.set_xticklabels(["Original", "Rauschen", "Rekonstruiert", "Fleck", "Fleck rek."])
ax.set_yticks([])
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Autoencoder und Generierung**</font>


In this lecture, you learned to:
- Unterscheiden diskriminative und generative Aufgaben anhand kleiner Datenbeispiele. 
- Trainieren kleine Autoencoder zur Rekonstruktion von Digits- oder MNIST-Teilmengen. 
- Analysieren Rekonstruktionsfehler, Denoising, latente Interpolation und Grenzen generativer Ansätze. 

In the next Module (Module 20), we will go over 'Abschlussprojekt'